# Description Deep Dive: Honest Summaries and Visualization

**Topic 06 · 1 lecture**

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/honr_464_logo.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb06_description_student.ipynb)

---

## 🧭 Inquiry & Claim Boundary

**Inquiry emphasis:** **Description**, the first compass position (descriptive
kind · the data at hand). In plain terms: a descriptive question asks what is
actually in the records in front of you. Example: "in these 10,000 interviews,
what share report high trust in the police?" Every project touches this
position. Before you can generalize, predict, or reason about cause, you have
to say plainly what your data contain. Doing that honestly is harder than it
looks. A summary compresses thousands of answers into a few numbers, and every
compression can distort. This notebook is also the course's anchor for the
**explore → declare → confirm loop**, the discipline that turns a pattern you
merely spotted into a claim you are actually allowed to make.

| | |
|---|---|
| **A claim this topic PERMITS** | "In these 10,000 resampled LAPOP Brazil interviews, X% report Y, and the answers pile up in shape Z." |
| **A claim this topic does NOT permit** | "Brazilians believe Y" (that reaches past the data) or "Y because Z" (that claims a cause). Description stops at the data's edge. |

**Where this sits in the course:** Phase 3, data and answer strategies. This is
the first of the four compass deep dives. It develops milestone M04, your
Measurement Plan, presented at the Friday studio, and the Table 1 skeleton you
draft today feeds M05, your Data Strategy.

*Provenance: RDSS ch. 15 'Observational: descriptive' (+ ch. 2 declaration idea) + rdss lapop_brazil | observational descriptive designs; the explore→declare→confirm loop | dataset shipped as CSV (MIT, attributed); 10k-resample caveat taught; loop named and drilled | adapted + newly-constructed-from-source-concept*

## Learning Objectives

By the end of this notebook, you will be able to:

1. Name a variable's type (nominal, ordinal, or numeric) and choose a summary
   that fits it.
2. Read the full shape of a variable's answers, and say what the shape reveals
   that the average hides.
3. Report a typical value together with how much the answers vary, and run one
   honest group comparison that stops at the data's edge.
4. Spot and repair the two classic chart distortions: the truncated axis and
   the cherry-picked frame.
5. Run one full turn of the explore → declare → confirm loop, testing a
   finding on rows the search never touched.
6. Draft the Table 1 skeleton for your own project's future data.

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

## 1. Why This Matters: the Provenance Caveat You Owe Every Reader

> *"Show me your Table 1 before you show me your finding. If you cannot
> describe your data honestly, if you cannot tell me what is in it and who is
> in it, I will not believe anything you built on top of it."*
> — your **thesis advisor**, on the first draft they will ever read

Description is where a reader's trust is won or lost. Every fancy method
downstream inherits whatever distortion you let into the plain summary. So the
discipline starts with two habits. Say what is in the data. Say where the data
came from.

You will practice on a real survey: 10,000 interviews from the
AmericasBarometer (LAPOP) study of Brazil, bundled with the course textbook.
One catch comes first, and the catch is itself the first lesson.

> **A question that often comes up here:** *"If this is real survey data, why
> can't I just say what Brazilians think?"* Because this file is a
> **10,000-row resample with replacement** of the original survey. In plain
> terms: the package built it by drawing rows from the real interviews over
> and over, like pulling names from a hat and dropping each name back in after
> every draw. That makes the file perfect for learning and planning, because
> the patterns are realistic and the code runs fast. It also makes the file
> dishonest for substantive claims, because the resampling manufactures a
> precision the original survey never had. So the caveat travels with every
> number you quote from it. Carrying that caveat along is what "documenting
> provenance" means in practice.

---

## 2. Variable Types and Honest Summaries

**Guiding question:** *before you summarize a column, how do you know which
summaries the column will even let you compute honestly?*

Every dataset shares the same anatomy. The rows are the units you observed.
The columns are the variables you recorded. Each cell holds one observation:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/variables_observations.png" width="75%"/></center>

*The anatomy of a data table: rows are the units you observed, columns are the
variables you recorded, and each cell is one observation. (From Professor
Moreira's QM 67000 Business Analytics slides.)*

A summary only tells the truth if it fits the variable. Average a category
code and you get a number that looks fine and means nothing. Three types cover
almost everything you will meet:

- **Nominal**: labels with no order, like a municipality or a party.
  Summarize with counts and the most common category, never with a mean.
- **Ordinal**: ordered categories with unequal or unknown gaps, like a 1–7
  trust scale. Counts are safe, and so is the **median**, the middle answer
  once you line them all up. The mean is common but leans on the fiction that
  the gaps are equal.
- **Numeric**: genuine quantities with meaningful gaps, like age or income.
  Mean and median both apply.

> **A question that often comes up here:** *"Everyone averages their 1–7
> scales. Can't I just treat an ordinal item as numeric?"* You can, and much
> of social science does, but name what the shortcut assumes. Taking a mean
> treats the step from "1" to "2" as identical to the step from "6" to "7",
> which an ordinal scale never promised. The safe habit is to report the
> median and the shape whenever the mean would lean hard on that equal-gaps
> fiction, and to say so out loud on the occasions you use the mean anyway.

Load the file and read its types off the data itself.

> 💡 **Gemini Prompt:** "This cell loads a survey dataset and then prints, for
> every column, its stored data type and its number of distinct values: [paste
> the next cell]. Explain why the count of distinct values is only a HINT to
> whether a variable is nominal, ordinal, or numeric, and give me one case
> where that hint would mislead me."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check that the load prints 10,000 rows and 10 columns. The shape assert
>   guards it, so confirm the ✓ line appears.
> - [ ] Pick one column and decide its type from what it MEASURES, not from its
>   distinct-value count. Then see whether Gemini's guess matches your reasoning.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Load the teaching resample and confirm its shape before trusting anything.
lapop = load_course_data("lapop_brazil.csv")
assert lapop.shape == (10000, 10), "unexpected shape — flag this!"
print("✓ Loaded lapop_brazil.csv —", lapop.shape[0], "rows,", lapop.shape[1], "columns")
print()
print("Columns and stored dtypes:")
print(lapop.dtypes.to_string())
print()
print("Distinct values per column (a fast type hint):")
print(lapop.nunique().to_string())

### 🔍 Reading the Evidence

The `nunique` counts are a first clue to type, not a verdict. In the cell
below, answer three things. First: `municipality` has 107 distinct values and
`trust_police` has 7. Which is **nominal** and which is **ordinal**, and how
do the counts help you tell? Second: `self_efficacy_political` has only 4
distinct values. Does a small count of distinct values *prove* a variable is
categorical? (Careful: a numeric age rounded to decades would also show few
values.) Third: name one column where computing a mean would produce a summary
that lies, and say why.

### YOUR ANSWER HERE:

**Nominal vs ordinal (municipality vs trust_police), and the tell:**

**Does few-distinct-values prove categorical?**

**A column where the mean would lie, and why:**

---

### 📝 Practice: repair three flawed descriptive claims

Each claim below is stated as if the data permit it. They do not. In the cell
that follows, repair each one so it says exactly what a descriptive summary of
`lapop_brazil` could support, and no more. Watch for three failure modes:
sneaking past the data's edge (generalization), smuggling in cause, and
forgetting the resample caveat.

- **A.** "Brazilians trust the military more than the police."
- **B.** "Trust in the supreme court is low because the courts are corrupt."
- **C.** "The average municipality in Brazil scored 4.4 on trust in police."


### YOUR ANSWER HERE:

**A repaired:**

**B repaired:**

**C repaired:**

---

### 🔮 Pause & Predict

A variable's type tells you which summaries are *legal*. Even a legal summary
can hide the most important thing in the data, and the next chart shows how.
You are about to plot the full **distribution** of `trust_military`. A
distribution is the full lineup of answers: how many of the 10,000 people
chose 1, how many chose 2, and so on up to 7. This variable's average is a
shade above 5. **Before running the next cell**, sketch in words the shape you
expect. A tidy hump centered near 5? A pile at the high end? A split into
camps? Commit to one picture and one sentence of reasoning below.

### YOUR ANSWER HERE:

**The shape I expect (in words):**

**Why:**

---

In [ ]:
# The reveal — run AFTER committing your prediction above.
counts = lapop["trust_military"].value_counts().sort_index()

fig, ax = plt.subplots()
ax.bar(counts.index, counts.values, color="#4C72B0", edgecolor="white")
ax.set_xlabel("Trust in the military (1 = none, 7 = a lot)")
ax.set_ylabel("Number of respondents")
ax.set_title("Distribution of reported trust in the military\n(lapop_brazil teaching resample, n = 10,000)")
mean_tm = lapop["trust_military"].mean()
ax.axvline(mean_tm, color="#C44E52", linestyle="--", linewidth=2,
           label=f"mean = {mean_tm:.2f}")
ax.legend()
plt.tight_layout()
plt.show()

print("Counts per scale point:")
print(counts.to_string())
print(f"\nmean = {mean_tm:.2f}   median = {lapop['trust_military'].median():.0f}   "
      f"mode = {lapop['trust_military'].mode().iloc[0]:.0f}")

In [ ]:
# Self-check: the mean sits in a valley, not on a peak — the point of this section.
assert abs(lapop["trust_military"].mean() - 5.16) < 0.01, "mean drifted — investigate"
assert lapop["trust_military"].mode().iloc[0] == 7, "mode should be 7"
assert counts.loc[5] < counts.loc[7], "expected far more 7s than 5s"
print("✓ Self-check passed: mean ≈ 5.16 lands between the scale points, "
      "mode is 7, and 7 is by far the most common answer.")

### 🔍 Reading the Evidence

The average is about 5.16. Now look where 5.16 actually falls: almost nobody
sits there. The single most common answer is 7, with over 3,500 people. There
is a second pile at 6 and a stubborn lump of low-trust answers at 1. The mean
is a single number, so it splits the difference and lands in a valley between
the camps. In the cell below, write two things: what the *shape* tells a
reader that the mean of 5.16 hides, and one decision a researcher might get
wrong by reporting only the average.

### YOUR ANSWER HERE:

**What the shape shows that the mean hides:**

**A decision the average alone could get wrong:**

---

## 3. Center, Spread, and One Honest Group Comparison

**Guiding question:** *when is an average a fair summary of a group, and when
is it a mask laid over two camps that have nothing in common?*

Every honest numeric summary reports two things together. The **center** is
the single value that stands in for a typical answer, like the mean or the
median. The **spread** is how far the answers scatter around that typical
value. Example: two dorms can both average 5 hours of sleep while one dorm
mostly sleeps about 5 and the other splits between all-nighters and 9-hour
sleepers. A mean with no spread is a half-truth.

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/spread_vs_center.png" width="75%"/></center>

*Two groups with the same mean but very different spread. A center reported
without its spread is only half the summary. (From Professor Moreira's QM
67000 Business Analytics slides.)*

Below you split respondents by ideological self-placement (a 1–10 scale,
where 1–5 counts as "left" and 6–10 as "right") and compare average trust in
the military. This is a descriptive group comparison: real, useful, and
sharply bounded.

> **A question that often comes up here:** *"If I already report the mean, why
> does the spread matter so much?"* Because two groups can share an identical
> mean and be nothing alike. One bunches tightly around it while the other
> pulls apart into extremes, and a decision that treats them as the same group
> would fail for one of them. The spread tells your reader whether the average
> describes a crowd standing together or the empty midpoint between two camps,
> which is exactly the trap the trust-in-military chart just sprang.

### 🛠️ Run the Study: center, spread, and a two-group comparison

Run the cell. It reports the center and spread of `trust_military` overall,
then the same numbers by ideological group. Two spread measures appear: the
**standard deviation** (roughly, the typical distance of an answer from the
mean) and the **IQR** (the range that holds the middle half of the answers).
Read the numbers before you read the next markdown cell.

> 💡 **Gemini Prompt:** "This cell reports the mean, median, standard
> deviation, and interquartile range of a trust item overall, then splits
> respondents into left and right ideological groups and compares their means:
> [paste the next cell]. Explain why a group mean reported WITHOUT its spread
> is only half a summary, and what the standard deviation adds to a two-group
> comparison."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the two group means against the printed difference. Does right
>   minus left really equal the gap the cell reports?
> - [ ] Confirm each group's standard deviation prints alongside its mean; a
>   comparison of centers with no spreads is exactly the half-truth to avoid.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Center + spread overall, then a two-group descriptive comparison.
tm = lapop["trust_military"]
print("Trust in the military — overall")
print(f"  mean   = {tm.mean():.2f}")
print(f"  median = {tm.median():.0f}")
print(f"  std    = {tm.std():.2f}   (spread around the mean)")
print(f"  IQR    = {tm.quantile(.25):.0f} to {tm.quantile(.75):.0f}")
print()

lapop["ideo_group"] = np.where(lapop["ideology"] <= 5, "left (1–5)", "right (6–10)")
by_group = lapop.groupby("ideo_group")["trust_military"].agg(["mean", "std", "count"]).round(2)
print("Trust in the military — by ideological self-placement")
print(by_group.to_string())
gap = (by_group.loc["right (6–10)", "mean"] - by_group.loc["left (1–5)", "mean"])
print(f"\nDifference in group means (right − left): {gap:+.2f} points on the 1–7 scale")

In [ ]:
# Self-check: the comparison is real and its direction is stable.
assert abs(by_group.loc["left (1–5)", "mean"] - 4.78) < 0.01, "left mean drifted"
assert abs(by_group.loc["right (6–10)", "mean"] - 5.54) < 0.01, "right mean drifted"
assert by_group.loc["right (6–10)", "mean"] > by_group.loc["left (1–5)", "mean"]
print("✓ Self-check passed: right-leaning respondents report higher average "
      "military trust (≈5.54) than left-leaning respondents (≈4.78) in these data.")

### 🔍 Reading the Evidence: where this claim must stop

The gap is about 0.77 points, and it is real *in these data*. Now patrol its
edge. In the cell below, write the **most precise claim** this comparison
permits. It must name the resample and the scale. Then write the two tempting
sentences you must NOT write: one that generalizes to all Brazilians, and one
that treats the gap as proof that ideology *causes* trust. For each forbidden
sentence, say in one line what evidence it would need that a description
cannot supply.

### YOUR ANSWER HERE:

**Most precise permitted claim:**

**Forbidden generalization (and what it would need):**

**Forbidden causal claim (and what it would need):**

---

### ⚖️ Make a Design Choice: which center do you report?

`govt_pride` is a 1–7 pride-in-government scale, heavily piled at the low
end, with a mode of 1 (the single most common answer) and a long tail upward.
You must summarize it in one number for a table. **Choose one:** report the
**mean**, or report the **median**. In the cell below, first recompute both
numbers yourself, then defend your choice in a short paragraph. Say what your
chosen number tells the reader, what it hides, and why the other number would
mislead more for this particular shape.

### YOUR ANSWER HERE (recompute here, then defend your choice):

```python
# Recompute both — do not take anyone's word for it.
print(lapop["govt_pride"].mean(), lapop["govt_pride"].median())
```

**My choice (mean or median) and its defense:**

---

## 4. Visualization That Tells the Truth

> *"A chart is an argument. The reader's eye does the reasoning before their
> brain catches up. If the picture overstates the numbers, you have made a
> claim you cannot defend, and you made it faster than anyone could check."*
> — a **journal reviewer** who rejects more figures than sentences

A distorted chart is not usually a fake chart. It is an *accurate* chart drawn
to make the eye conclude more than the numbers support. The most common trick
is the **truncated axis**: start the y-axis partway up the scale, and a small
real difference balloons into a dramatic one. Add a **cherry-picked framing**,
meaning a loaded title or a convenient comparison, and the picture argues a
case the data never made.

You will build a deliberately misleading chart from a real `lapop_brazil`
difference, then repair it. Same numbers, both times. Only the honesty
changes.

In [ ]:
# Build the two group means we will chart (govt_pride by ideological group).
by_pride = lapop.groupby("ideo_group")["govt_pride"].mean()
left_val = by_pride.loc["left (1–5)"]
right_val = by_pride.loc["right (6–10)"]
print(f"Average government pride (1–7 scale):")
print(f"  left (1–5):  {left_val:.2f}")
print(f"  right (6–10): {right_val:.2f}")
print(f"  real difference: {right_val - left_val:+.2f} points on a 7-point scale")

assert abs(left_val - 2.68) < 0.01 and abs(right_val - 3.29) < 0.01, "means drifted"
print("✓ Both group means below the scale midpoint of 4 — government pride is low in both groups.")

Both group means sit below the scale midpoint of 4, and the real gap is about
0.62 points on a 7-point scale. Modest. Now watch how a drawing can sell that
modest gap as a chasm.

> 💡 **Gemini Prompt:** "This cell charts two group means that differ by about
> 0.62 on a 1–7 scale, but it starts the y-axis at 2.5 and gives the chart a
> punchy title: [paste the next cell]. Explain, step by step, how starting the
> axis above zero turns a small real gap into a bar that looks several times
> taller, and how the printed exaggeration factor is computed."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the printed exaggeration factor against the bars you see. Does
>   the right bar really look several times the left, for a real gap of only
>   0.62?
> - [ ] Confirm both group means sit below the scale midpoint of 4, so the
>   "surges" title claims more than the numbers support.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# The DISTORTED chart — this is how a 0.62-point gap gets sold as a chasm.
fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(["Left", "Right"], [left_val, right_val],
       color=["#8899AA", "#C44E52"], edgecolor="white")
ax.set_ylim(2.5, 3.4)                       # truncated axis: the whole trick
ax.set_ylabel("Government pride")           # no scale range shown
ax.set_title("Government pride SURGES on the political right")  # cherry-picked framing
plt.tight_layout()
plt.show()

# How big is the visual lie? Compare bar heights ABOVE the truncated floor.
floor = 2.5
exaggeration = (right_val - floor) / (left_val - floor)
print(f"Truncated at {floor}: the right bar is drawn {exaggeration:.1f}× taller "
      f"than the left — for a real gap of only {right_val - left_val:.2f} points.")

### 🛠️ Hands-On: repair the chart, one distortion at a time

Now un-distort it. The next cell redraws the *same two numbers* honestly,
fixing three things at once: the axis runs the full 1–7 scale the variable
actually lives on; a reference line marks the scale midpoint, so "high" and
"low" are visible; and the title describes instead of editorializing, while
the caption carries the resample caveat. Run it, then put the two pictures
side by side.

In [ ]:
# The HONEST chart — same data, truthful drawing.
fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(["Left (1–5)", "Right (6–10)"], [left_val, right_val],
       color=["#8899AA", "#4C72B0"], edgecolor="white")
ax.set_ylim(1, 7)                                  # full scale, no truncation
ax.axhline(4, color="gray", linestyle=":", linewidth=1.5, label="scale midpoint (4)")
ax.set_ylabel("Average government pride (1 = none, 7 = a lot)")
ax.set_title("Average government pride by ideological self-placement")
ax.legend(loc="upper right")
ax.text(-0.4, 0.4, "lapop_brazil teaching resample, n = 10,000 — for planning, not population claims",
        fontsize=8, color="gray")
plt.tight_layout()
plt.show()

print(f"Same numbers ({left_val:.2f} vs {right_val:.2f}) — now the reader sees "
      f"that both groups report LOW pride, and the gap is small.")

### 🔍 Reading the Evidence: the impression is also a claim

Look at the two figures side by side. The numbers never changed. In the cell
below, write three things: what *visual* claim the first chart makes that the
numbers do not support; which single repair (axis, framing, or context) did
the most to defuse it; and the uncomfortable one: whether a technically
accurate but misleading chart counts as a lie, and who is responsible when a
reader is fooled, the maker or the reader.

### YOUR ANSWER HERE:

**The unsupported visual claim in the distorted chart:**

**The repair that mattered most:**

**Is an accurate-but-misleading chart a lie? Who is responsible?**

---

### The honest-visualization checklist

Keep this next to every figure you make for the rest of the course:

- **Axis:** does it start where the scale starts, so bar heights stay
  proportional to the numbers?
- **Framing:** does the title *describe* the data rather than argue a verdict?
- **Context:** are the sample size, the scale range, and any resampling
  visible on or beside the figure?
- **Comparison:** is the comparison the fair one, not the flattering one?
- **Re-drawability:** could a careful reader rebuild your chart from your
  numbers and get the same picture?

> **A question that often comes up here:** *"Is it always wrong to start a bar
> chart's axis above zero?"* For bar charts, almost always. A bar encodes its
> value by length, so a truncated baseline makes the length itself lie. The
> honest exception is a line chart tracking change over time, where the reader
> judges the slope rather than compares lengths, and a zoomed axis can reveal
> a real trend without distorting it. The test is never the axis alone. It is
> whether the visual quantity the eye compares still matches the numbers.

---

## 5. The Explore → Declare → Confirm Loop

Look back at the last hour. You loaded a file, read its types, plotted a
distribution, compared groups, and repaired a chart. Every bit of it was
**exploration**: a disciplined descriptive pass over the data at hand. That
work is essential, and it hides a trap. Naming the trap is the job of this
final section.

Here is the trap. When you rummage through a dataset and report the biggest,
cleanest pattern you find, you have let the data pick your claim. A pattern
picked *because* it was the strongest of many you looked at will, on average,
look weaker the next time anyone checks. An exploratory finding is a
candidate claim, not a confirmed one. It earns claim status in two more
steps. You **declare** it as a precise inquiry, the exact quantity stated in
words, so it can no longer shift to fit the data. Then you **confirm** it
with a **held-out check**: you set part of the data aside, never touch it
while exploring, and test the declared finding on that untouched part.
Example: search one half of the interviews for a pattern, then recompute it
on the other half.

That three-beat rhythm, explore → declare → confirm, is the loop this course
returns to again and again. Run one full turn of it now, on the very survey
you just spent the hour describing.

> **A question that often comes up here:** *"If I already found the pattern,
> why would checking it on other rows change anything?"* Because the half you
> explored handed you the winner of a search, the biggest gap out of several
> you compared, and searches reward luck as well as signal. On a fresh half of
> the data, the lucky part of a gap tends to shrink back while the real part
> stays. If the finding survives that second look, you can report it with a
> much straighter face. If it collapses, you just caught yourself before your
> reader would have.

---

### 🛠️ Run the Loop: explore one half, declare, then confirm on the other

Run the cell. It splits the 10,000 interviews into two random halves. On the
**explore** half it compares left- versus right-leaning respondents across six
survey items and keeps the single biggest gap. Then it **declares** that
finding as one precise sentence. Finally it **confirms** the finding by
recomputing the exact same quantity on the **confirm** half, which had no say
in which item won.

> 💡 **Gemini Prompt:** "This cell splits a survey into two random halves,
> searches the first half for the biggest left-versus-right gap among several
> items, declares that gap as a finding, then recomputes it on the second
> half: [paste the next cell]. Explain why searching many comparisons and
> reporting the biggest one can overstate a pattern, and why checking it on
> the held-out half is a more honest test than reporting the explore-half
> number alone."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Confirm the confirm-half gap is computed on rows the search never
>   touched. The whole point is that these rows had no hand in choosing the
>   winning item.
> - [ ] Compare the two gap numbers yourself. Did the finding hold its
>   direction and roughly its size, or did it shrink? Commit to an answer
>   before trusting Gemini's.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# One full turn of the loop: EXPLORE -> DECLARE -> CONFIRM on split halves.
idx = rng.permutation(len(lapop))            # a reproducible shuffle (SEED = 464)
half = len(lapop) // 2
explore = lapop.iloc[idx[:half]].copy()      # the half we are allowed to rummage in
confirm = lapop.iloc[idx[half:]].copy()      # the half we will not touch until the end
print(f"Split: explore n = {len(explore)}, confirm n = {len(confirm)}")

def right_minus_left(df, item):
    """Mean gap on `item`: right-leaning (ideology 6-10) minus left (1-5)."""
    left  = df.loc[df["ideology"] <= 5, item].mean()
    right = df.loc[df["ideology"] >= 6, item].mean()
    return right - left

# EXPLORE: compare several candidate items, keep the biggest gap.
items = ["trust_police", "trust_military", "trust_supreme_court",
         "govt_pride", "govt_responsive", "support_political_system"]
gaps = {it: right_minus_left(explore, it) for it in items}
winner = max(gaps, key=lambda it: abs(gaps[it]))
print("\nEXPLORE half — right-minus-left gap for each candidate item:")
for it in items:
    print(f"  {it:26s} {gaps[it]:+.2f}")
print(f"\nBiggest gap the search found: {winner} ({gaps[winner]:+.2f} points)")

# DECLARE: pin the finding as one precise descriptive inquiry, in words.
print(f"\nDECLARED inquiry: 'In these interviews, right-leaning respondents "
      f"(ideology 6-10) report higher\n  average {winner} than left-leaning "
      f"respondents (1-5).' -> quantity = right-minus-left mean gap.")

# CONFIRM: recompute the SAME quantity on the untouched half.
explore_gap = gaps[winner]
confirm_gap = right_minus_left(confirm, winner)
print(f"\nCONFIRM the declared quantity ({winner}):")
print(f"  explore half (where we found it): {explore_gap:+.2f}")
print(f"  confirm half (never searched)   : {confirm_gap:+.2f}")
held = "held up" if np.sign(confirm_gap) == np.sign(explore_gap) else "did NOT hold"
print(f"\nOn rows the search never saw, the finding {held}.")

In [ ]:
# Self-check: SEED fixes the split, so the winner and both gaps are reproducible.
assert winner == "support_political_system", "biggest explored gap moved — is SEED = 464?"
assert round(explore_gap, 2) == 0.90 and round(confirm_gap, 2) == 0.86, "gaps drifted"
print("✓ Self-check passed: support_political_system had the biggest explored gap "
      "(+0.90);\n  on the untouched half it came in at +0.86 — same direction, "
      "nearly the same size.")

### 🔍 Reading the Evidence: an exploratory finding is not a confirmed claim

The gap you *found* was +0.90. On rows the search never saw, it came back at
+0.86, the same direction and nearly the same size, so this particular finding
held up. That is the happy case. Pause on the unhappy one: if the confirm
number had collapsed toward zero, the honest move would be to report *that*,
not the flattering +0.90 you explored your way into.

Two connections carry this loop into the rest of the course:

- **Exploration that shaped what you report is part of your procedure.** You
  did not compute one number; you searched six and reported the winner. That
  search is part of your answer strategy, and honesty requires saying so. RDSS
  states the rule as "the answer strategy is the whole procedure" (§9.1.3),
  and nb09 drills it in full. Reporting the best of many explored patterns as
  though it were the only comparison you ever ran is the quiet way descriptive
  work overreaches.
- **A pattern you stumble on late is a new inquiry, not a free finding.** If
  an unplanned pattern jumps out near the end of a project, treat it as a
  pivot: declare it fresh and, where you can, confirm it on new data, rather
  than bolting it onto the claims you set out to make (RDSS ch. 22; this
  returns in nb17).

The boundary to carry out the door: **an exploratory finding is not a
confirmed claim.** Explore freely, because that is how good questions are
born, but let nothing cross into "what I found" until it has been declared
and, wherever possible, confirmed on data that had no hand in suggesting it.

---

### 🎯 Project Transfer: your Table 1 skeleton

Every credible empirical paper opens with a "Table 1" that describes the
sample before any finding appears. Start yours now, as part of your M04
Measurement Plan. In the cell below, sketch the skeleton for *your* project's
future data. List 4 to 6 variables you expect to collect. For each, name its
**type** and the **summary** you will report (count / mean+sd / median+IQR).
For your one key outcome, write the **claim boundary** your description will
stop at. Then add one figure pledge: name the figure your project will most
likely need, the checklist item it is most at risk of failing, and the
drawing decision that will keep it honest. You are not filling in numbers yet.
You are committing to how you will describe honestly once numbers exist, and
that commitment feeds straight into your M05 Data Strategy.

### YOUR ANSWER HERE:

| Variable | Type (nominal/ordinal/numeric) | Summary I will report | Claim boundary |
|---|---|---|---|
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |

**My key outcome and the exact sentence my description will stop at:**

**My figure pledge (the figure, its riskiest checklist item, my safeguard):**

---

### 🎟️ Claim Ticket

**Claim Ticket #1** — write your best descriptive claim from today's data,
worded to stop *exactly* at the data's edge. It must name the resample, report
a center *and* a spread (or a distribution shape), and contain no "plausibly"
and no "because".

### YOUR ANSWER HERE:

**My edge-of-the-data descriptive claim, one sentence:**

---

## 6. If You Want to Go Further *(optional)*

Everything below is optional enrichment. Nothing in it is required for M04 or
the Friday studio.

### Shape at a glance: the boxplot

The bar chart showed you the full shape of trust in the military. A
**boxplot** is the compact version of that same shape: it marks the median and
the middle half of the answers, so a skew you saw in the bars turns into a
lopsided box. Learn to read the two together:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/shape_boxplot_map.png" width="78%"/></center>

*How a distribution's shape maps onto a boxplot: the same skew, read two ways.
(From Professor Moreira's QM 67000 Business Analytics slides.)*

### One more distortion to know: the pie chart

A truncated axis exaggerates a gap; a pie chart tends to hide one. People
compare *lengths* far more accurately than they compare angles or areas, so
slices that are easy to draw are hard to read. A 3-D tilt makes it worse by
throwing the front slice toward the viewer. The same numbers drawn as a bar
chart hand the eye the comparison a pie only asks it to guess:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/pie-vs-bar.png" width="80%"/></center>

*The same numbers drawn as a pie and as a bar chart: the bar lengths are easy
to compare, the pie slices are not. (From Professor Moreira's QM 67000
Business Analytics slides.)*

The most famous case is a 2008 Apple product keynote:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/steve-jobs-pie.jpg" width="70%"/></center>

*A 2008 Apple keynote 3-D pie chart: the 19.5% slice tilted toward the front
is drawn to look nearly as large as the 39% slice behind it, perspective
distorting the comparison. (Apple keynote photograph, 2008, a classic
visualization-criticism case, used in Professor Moreira's slides.)*

For a public gallery of axis and dataviz distortions to study, the *Calling
Bullshit* project keeps an open case index and tools page:
[callingbullshit.org/tools](https://callingbullshit.org/tools/) and
[callingbullshit.org/case_studies.html](https://callingbullshit.org/case_studies.html).

### 📝 Practice: name the distortion

Three figures are described below. For each, name the distortion mechanism
(from the checklist) and the one repair that would make it honest. Write your
answers in the cell that follows.

- **A.** A bar chart of two campus dining halls' satisfaction (4.2 vs 4.4 out
  of 5) with a y-axis running from 4.0 to 4.5, titled "Hillenbrand crushes the
  competition."
- **B.** A histogram of exam scores whose bins are 0–59, 60–61, 62–63, and
  64–100, making the middle look like a towering spike.
- **C.** A line chart of one club's membership over the three months it
  happened to grow, cropped from a two-year record in which it mostly fell.


### YOUR ANSWER HERE:

**A (distortion + repair):**

**B (distortion + repair):**

**C (distortion + repair):**

---

## 7. Wrap-Up

Today you did the unglamorous, load-bearing work of Description, the first
compass deep dive. You matched summaries to variable types, read a
distribution instead of trusting its average, reported center with spread,
ran one honest group comparison, repaired a truncated-axis chart back into a
truthful picture of the same numbers, and closed with one full turn of the
explore → declare → confirm loop. Underneath all of it ran one discipline:
description stops at the data's edge, and the resample caveat rides along
with every number you quote.

> **"A description is honest when a careful reader, given only your summary,
> would picture the data the way it actually is — no larger, no cleaner, and no
> more certain than it earned."**

Next, nb07 opens MIDA's Data strategy and asks the question description
cannot: *whose* data is this? Every dataset was drawn from some **sampling
frame**, the concrete list of people or cases a study could actually reach,
like a registrar's list of enrolled seniors standing in for "college kids
everywhere." You will draw real samples from a voter file, watch a
convenience sample lie, and start the M05 Data Strategy that decides who gets
into your evidence in the first place. Bring your Table 1 skeleton. It is
about to meet its sampling frame.

---

## 8. Sources & Provenance

**Provenance of this notebook:**
- *RDSS ch. 15 'Observational: descriptive' | observational descriptive designs | variable types, distributions, group summaries, the descriptive claim boundary | adapted (concept-level, honors non-quant audience)*
- *RDSS ch. 15 + ch. 2 (declaration) + §9.1.3 + ch. 22 | the explore → declare → confirm loop | a split-half demonstration: search one half, declare the finding, confirm on the untouched half; loop named as this course's anchor | newly-constructed-from-source-concept*
- *lapop_brazil.csv | rdss package data | trust/pride/ideology items summarized, charted, and split-confirmed | adapted (10,000-row resample-with-replacement; caveat taught explicitly as a provenance lesson)*
- *fresh | honest-visualization checklist + truncated-axis repair | n/a | newly-constructed-from-source-concept*
- *callingbullshit.org (public index) | misleading-axes / dataviz material | linked as optional study, index pages only | referenced*

**Dataset attribution:** Dataset from the `rdss` package (Blair, Coppock &
Humphreys, MIT License), companion to *Research Design in the Social Sciences*
(2023).

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the
  Social Sciences*, ch. 15 'Observational: descriptive'. Free online:
  [book.declaredesign.org](https://book.declaredesign.org/).
- *(Optional)* Bergstrom, C. & West, J. *Calling Bullshit*, misleading axes /
  dataviz case index: [callingbullshit.org/tools](https://callingbullshit.org/tools/).

---

<center>

Thank you!

</center>